In [1]:
import math
import os
import time
import re
import requests
import polars as pl

from decorator import append
from dotenv import load_dotenv
from duckdb.experimental.spark.sql.functions import length
from datetime import datetime
from utils.postgres.postgres_connection import PostgresConnection


load_dotenv(override=True)

RIOT_PERSONAL_API_KEY = os.getenv("LOL_PERSONAL_API_KEY")
POSTGRES_LOCAL_URL = os.getenv("POSTGRES_LOCAL_URL")
connection_wrapper = PostgresConnection(POSTGRES_LOCAL_URL)

In [2]:
_camel_case_regex = re.compile(
    r"(?<=[a-z])(?=[A-Z])"  # camelCase: survive|Min
    r"|(?<=[A-Z])(?=[A-Z][a-z])"  # acronym: HTML|Parser
    r"|(?<=[a-zA-Z])(?=\d)"  # letter→digit: survive|15
    r"|(?<=\d)(?=[a-zA-Z])"  # digit→letter: 15|min
)


def camel_to_snake(input_str: str) -> str:
    return _camel_case_regex.sub("_", input_str).lower()


def snake_all_keys(json_obj: dict):
    if isinstance(json_obj, dict):
        return {camel_to_snake(key): snake_all_keys(value) for key, value in json_obj.items()}

    if isinstance(json_obj, list):
        return [snake_all_keys(value) for value in json_obj]

    return json_obj


In [3]:
players = pl.read_database_uri("SELECT * FROM players", POSTGRES_LOCAL_URL)
players

puuid,game_name,tag_line,region,synced_at
str,str,str,str,datetime[μs]
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…","""bird raven""","""0000""","""EUN1""",2026-05-31 18:10:37.551072
"""QMPPN4oUrY-FW-BBOZ34yDZZmjghPa…","""KaizokuNoOni""","""2829""","""EUN1""",2026-05-31 18:10:37.860377
"""yf1OVyzGtKFWKAh__NIUFtnEisdohM…","""Nolba""","""EUNE""","""EUN1""",2026-05-31 18:10:38.176115
"""gw0udTm33m9cDPfENXndpjlXmkHwxl…","""Defenser""","""EUNE""","""EUN1""",2026-05-31 18:10:38.472933
"""tD8QYcsn9jmPaM7AACN9Id2fwgusfN…","""Dioklis""","""EUNE""","""EUN1""",2026-05-31 18:10:38.757716
…,…,…,…,…
"""dYiEC-ravKg74XNQMp4KsOp4S7aVlR…","""black nails""","""988""","""EUN1""",2026-05-31 19:15:39.289847
"""ZLw_1J27yiyrFC_UN7-Uck-IyYMBLL…","""AleEmotkaZostała""","""RMJ""","""EUN1""",2026-05-31 19:15:39.630180
"""0CPr6RzHVit2UWf0Cu0EXPbIj77eko…","""chiefelo""","""0314""","""EUN1""",2026-05-31 19:15:39.980448


In [25]:
from datetime import date


BASE_EUN1_URL = "https://eun1.api.riotgames.com"

headers = {
    'X-Riot-Token': RIOT_PERSONAL_API_KEY
}

def get_player_ranks(puuid: str):


    player_ranks_url=f"{BASE_EUN1_URL}/lol/league/v4/entries/by-puuid/{puuid}"
    response_ranks = requests.get(player_ranks_url,headers=headers)

    if response_ranks.status_code != 200:
        print("request failed with status code on getting player ranks", response_ranks.status_code)
        return response_ranks.status_code

    response_ranks_json = response_ranks.json()

    all_player_ranks = []

    for entry_ranks in response_ranks_json:
        player_ranks = {
            "puuid": puuid,
            "snapshot_date":date.today(),
            "queue_type": entry_ranks.get("queueType"),
            "tier": entry_ranks.get("tier"),
            "rank": entry_ranks.get("rank"),
            "league_points": entry_ranks.get("leaguePoints"),
            "wins": entry_ranks.get("wins"),
            "losses": entry_ranks.get("losses"),
            "hot_streak": entry_ranks.get("hotStreak"),
            "veteran": entry_ranks.get("veteran"),
            "fresh_blood": entry_ranks.get("freshBlood"),
            "inactive": entry_ranks.get("inactive"),
        }
        all_player_ranks.append(player_ranks)

    return all_player_ranks


get_player_ranks(players["puuid"][0])

[{'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'snapshot_date': datetime.date(2026, 6, 3),
  'queue_type': 'RANKED_SOLO_5x5',
  'tier': 'IRON',
  'rank': 'I',
  'league_points': 27,
  'wins': 5,
  'losses': 13,
  'hot_streak': False,
  'veteran': False,
  'fresh_blood': False,
  'inactive': False}]

In [29]:
player_ranks_arr = []

current_index = 0
for player in players.iter_rows(named=True):
    current_index += 1
    if current_index % 50 == 0:
        print(f"progress {math.floor(current_index / len(players) * 100)}%")

    try:
        player_ranks = get_player_ranks(player["puuid"])
        while player_ranks == 429:
            print("retrying after getting rate limited")
            time.sleep(121)
            player_ranks = get_player_ranks(player["puuid"])

        player_ranks_arr.extend(player_ranks)

    except Exception as e:
        print(f"Error fetching snapshot for {player['puuid']}: {e}")
        continue

player_ranks_arr


progress 1%
progress 3%
request failed with status code on getting player ranks 429
retrying after getting rate limited
progress 4%
progress 6%
request failed with status code on getting player ranks 429
retrying after getting rate limited
progress 8%
progress 9%
request failed with status code on getting player ranks 429
retrying after getting rate limited
progress 11%
progress 12%
request failed with status code on getting player ranks 429
retrying after getting rate limited
progress 14%
progress 16%
request failed with status code on getting player ranks 429
retrying after getting rate limited
progress 17%
progress 19%
request failed with status code on getting player ranks 429
retrying after getting rate limited
progress 20%
progress 22%
request failed with status code on getting player ranks 429
retrying after getting rate limited
progress 24%
progress 25%
request failed with status code on getting player ranks 429
retrying after getting rate limited
progress 27%
progress 29%
requ

[{'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'snapshot_date': datetime.date(2026, 6, 3),
  'queue_type': 'RANKED_SOLO_5x5',
  'tier': 'IRON',
  'rank': 'I',
  'league_points': 27,
  'wins': 5,
  'losses': 13,
  'hot_streak': False,
  'veteran': False,
  'fresh_blood': False,
  'inactive': False},
 {'puuid': 'QMPPN4oUrY-FW-BBOZ34yDZZmjghPaYoJqNongrmvIJJkD9MtNUvJG1FdgQcvywW23L_s_bA83HIWw',
  'snapshot_date': datetime.date(2026, 6, 3),
  'queue_type': 'RANKED_FLEX_SR',
  'tier': 'SILVER',
  'rank': 'IV',
  'league_points': 42,
  'wins': 3,
  'losses': 2,
  'hot_streak': False,
  'veteran': False,
  'fresh_blood': False,
  'inactive': False},
 {'puuid': 'QMPPN4oUrY-FW-BBOZ34yDZZmjghPaYoJqNongrmvIJJkD9MtNUvJG1FdgQcvywW23L_s_bA83HIWw',
  'snapshot_date': datetime.date(2026, 6, 3),
  'queue_type': 'RANKED_SOLO_5x5',
  'tier': 'IRON',
  'rank': 'I',
  'league_points': 60,
  'wins': 1,
  'losses': 4,
  'hot_streak': False,
  'veteran': False,
  

In [30]:
print(player_ranks_arr)

[{'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ', 'snapshot_date': datetime.date(2026, 6, 3), 'queue_type': 'RANKED_SOLO_5x5', 'tier': 'IRON', 'rank': 'I', 'league_points': 27, 'wins': 5, 'losses': 13, 'hot_streak': False, 'veteran': False, 'fresh_blood': False, 'inactive': False}, {'puuid': 'QMPPN4oUrY-FW-BBOZ34yDZZmjghPaYoJqNongrmvIJJkD9MtNUvJG1FdgQcvywW23L_s_bA83HIWw', 'snapshot_date': datetime.date(2026, 6, 3), 'queue_type': 'RANKED_FLEX_SR', 'tier': 'SILVER', 'rank': 'IV', 'league_points': 42, 'wins': 3, 'losses': 2, 'hot_streak': False, 'veteran': False, 'fresh_blood': False, 'inactive': False}, {'puuid': 'QMPPN4oUrY-FW-BBOZ34yDZZmjghPaYoJqNongrmvIJJkD9MtNUvJG1FdgQcvywW23L_s_bA83HIWw', 'snapshot_date': datetime.date(2026, 6, 3), 'queue_type': 'RANKED_SOLO_5x5', 'tier': 'IRON', 'rank': 'I', 'league_points': 60, 'wins': 1, 'losses': 4, 'hot_streak': False, 'veteran': False, 'fresh_blood': False, 'inactive': False}, {'puuid': 'yf1OVyzGtKFWK

In [39]:
player_ranks_df=pl.DataFrame(player_ranks_arr)
player_ranks_df = player_ranks_df.with_columns(
    pl.lit(date(2026, 6, 2)).alias("snapshot_date")
)
player_ranks_df

puuid,snapshot_date,queue_type,tier,rank,league_points,wins,losses,hot_streak,veteran,fresh_blood,inactive
str,date,str,str,str,i64,i64,i64,bool,bool,bool,bool
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…",2026-06-02,"""RANKED_SOLO_5x5""","""IRON""","""I""",27,5,13,false,false,false,false
"""QMPPN4oUrY-FW-BBOZ34yDZZmjghPa…",2026-06-02,"""RANKED_FLEX_SR""","""SILVER""","""IV""",42,3,2,false,false,false,false
"""QMPPN4oUrY-FW-BBOZ34yDZZmjghPa…",2026-06-02,"""RANKED_SOLO_5x5""","""IRON""","""I""",60,1,4,false,false,false,false
"""yf1OVyzGtKFWKAh__NIUFtnEisdohM…",2026-06-02,"""RANKED_SOLO_5x5""","""IRON""","""I""",71,8,11,true,false,false,false
"""gw0udTm33m9cDPfENXndpjlXmkHwxl…",2026-06-02,"""RANKED_SOLO_5x5""","""IRON""","""I""",12,1,4,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…
"""ZLw_1J27yiyrFC_UN7-Uck-IyYMBLL…",2026-06-02,"""RANKED_FLEX_SR""","""EMERALD""","""II""",52,16,19,false,false,false,false
"""0CPr6RzHVit2UWf0Cu0EXPbIj77eko…",2026-06-02,"""RANKED_SOLO_5x5""","""CHALLENGER""","""I""",1395,210,193,false,false,false,false
"""voxGacYDhQ3ptNab4tGe6ipPP3nZfc…",2026-06-02,"""RANKED_FLEX_SR""","""MASTER""","""I""",704,61,47,false,false,false,false


In [41]:
from utils.postgres.postgres_connection import PostgresConnection
connection_wrapper = PostgresConnection(POSTGRES_LOCAL_URL)

opening connection


In [42]:
connection_wrapper.upsert_dataframe_dto_safe(player_ranks_df, "ranks", ["puuid","queue_type","snapshot_date"])

0

In [36]:
player_ranks_df.write_parquet("ranks.parquet")